# Lesson 4: Abstractive Text Summarization

In this notebook, you manage challenges arising in large document summarization - input text can exceed model context lengths, generate hallucinated outputs, or trigger out-of-memory errors.

To mitigate these issues, this notebook demonstrates an architecture using prompt chunking and chaining with the [LangChain](https://python.langchain.com/docs/get_started/introduction.html) framework, a toolkit enabling applications leveraging language models.

You explore an approach addressing scenarios when user documents surpass token limits. Chunking splits documents into segments under context length thresholds before sequentially feeding them to models. This chains prompts across chunks, retaining prior context. You apply this approach to summarize call transcripts, meetings transcripts, books, articles, blog posts, and other relevant content.

## Task 4.1: Environment setup

In this task, you set up your environment.

In [48]:
import json
import os
import sys

from IPython.display import display, HTML

def show_output(output, type="md"):
    if type == "md":
        display(Markdown(output))
    else:
        display(HTML("<div style='white-space: pre-wrap; overflow-wrap: anywhere;'>"
                    + output
                    + "</div>"))

module_path = ".."
sys.path.append(os.path.abspath(module_path))

In [2]:
import boto3

bedrock_client = boto3.client(
    'bedrock-runtime',
    region_name=os.environ.get("AWS_DEFAULT_REGION")
)

## Task 4.2: Project Setup

In this section, we'll proceed with the initial configuration of the project.

In [37]:
from langchain_aws import ChatBedrock

modelId = "amazon.nova-micro-v1:0"
llm = ChatBedrock(
    model_id=modelId,
    model_kwargs={
        "maxTokenCount": 2048,
        "temperature": 0,
        "topP": 0.8
    },
    client=bedrock_client
)

> ChatBedrock is a chat model interface that works with structured messages and returns an AIMessage object. BedrockLLM is a traditional text-generation interface that accepts a string and returns a string.
Both can produce similar responses when using the same underlying model, prompt, and generation parameters. However, the output may differ due to input formatting, model integration, or generation variability.

## Task 4.3: Loading a markdown file with many tokens

In this task, you can find a markdown file with content about [Physics](../dataset/Physics.md). The following cell loads the text file and counts the number of tokens. You will see a warning indicating the number of tokens in the text file exceeds the maximum number of tokens for this model.

In [38]:
physics_doc = "../dataset/Physics.md"
physics_doc_content = None

with open(physics_doc, "r") as file:
    physics_doc_content = file.read()

llm.get_num_tokens(physics_doc_content)

7019

## Task 4.4: Splitting the long text into chunks

As you can notice in the output of the previous cell, the number of tokens is much larger than the established context window, with that we need to split the text into smaller chunks because it is too long to fit in the prompt. 

`RecursiveCharacterTextSplitter` in LangChain supports splitting long text into chunks recursively until the size of each chunk becomes smaller than `chunk_size`. A text is separated with `separators=["\n\n", "\n"]` into chunks, which avoids splitting each paragraph into multiple chunks.

Using 6,000 characters per chunk, you can get summaries for each portion separately. The number of tokens, or word pieces, in a chunk depends on the text.

The cell below splits the Markdown document into sections based on heading levels (`H1`, `H2`, and `H3`), preserving the heading hierarchy as metadata for each resulting document. This structure helps maintain semantic context during retrieval in a RAG pipeline.

In [39]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

documents = markdown_splitter.split_text(physics_doc_content)

The cell below counts the generated documents and calculates the number of tokens in the first document to evaluate the results of the Markdown splitting process.

In [40]:
num_documents = len(documents)

if documents:
    first_document_token_count = llm.get_num_tokens(
        documents[0].page_content
    )
    print(
        f"Number of documents: {num_documents}\n"
        f"Token count of the first document: {first_document_token_count}\n"
        f"Content of the first document: {documents[0].page_content}\n"
    )
else:
    print("No documents were generated.")

Number of documents: 23
Token count of the first document: 259
Content of the first document: Physics is the scientific study of matter, its fundamental constituents, its motion and behavior through space and time, and the related entities of energy and force. It is one of the most fundamental scientific disciplines. A scientist who specializes in the field of physics is called a physicist.  
Physics is one of the oldest academic disciplines. Over much of the past two millennia, physics, chemistry, biology, and certain branches of mathematics were part of natural philosophy, but during the Scientific Revolution in the 17th century, these natural sciences branched into separate research endeavors. Physics intersects with many interdisciplinary areas of research, such as biophysics and quantum chemistry, and the boundaries of physics are not rigidly defined. New ideas in physics often explain the fundamental mechanisms studied by other sciences and suggest new avenues of research in thes

## Task 4.5: Summarizing chunks and combining them

In this task, assuming that the number of tokens is consistent in the other documents, you should be good to go. You can use LangChain's `load_summarize_chain` to summarize the text. `load_summarize_chain` provides three ways of summarization: `stuff`, `map_reduce`, and `refine`.

- `stuff`: puts all the chunks into one prompt. Thus, this would hit the maximum limit of tokens.
- `map_reduce`: summarizes each chunk, combines the summaries, and summarizes the combined summary. If the combined summary is too large, it would raise an error.
- `refine`: summarizes the first chunk, and then summarizes the second chunk with the first summary. The same process repeats until all chunks are summarized.

Both `map_reduce` and `refine` invoke the LLM multiple times and take time for obtaining the final summary. Let's try `map_reduce` here.

In [41]:
from langchain.chains.summarize import load_summarize_chain

summary_chain = load_summarize_chain(llm=llm, chain_type="map_reduce", verbose=False)

> **Note:** Depending on your number of documents, Bedrock request rate quota, and configured retry settings - the chain below may take some time to run.

In [ ]:
# Debug cell

print(type(documents[0]))
print(type(documents[0].page_content) if hasattr(documents[0], "page_content") else "No page_content")
print(llm)
print(type(llm))

<class 'langchain_core.documents.base.Document'>
<class 'str'>
BedrockLLM
Params: {'model_id': 'amazon.nova-micro-v1:0', 'base_model_id': None, 'provider': 'amazon', 'stream': False, 'trace': None, 'guardrailIdentifier': None, 'guardrailVersion': None, 'maxTokenCount': 2048, 'topP': 0.8}
<class 'langchain_aws.llms.bedrock.BedrockLLM'>


In [ ]:
output = ""
output = summary_chain.invoke({ "input_documents": documents })

In [47]:
show_output(output['output_text'])

You have now experimented with using prompt chunking and chaining with the LangChain framework to summarize large documents while mitigating issues arising from long input text.

### Try it yourself
- Change the prompts to your specific usecase and evaluate the output of different models.
- Play with the token length to understand the latency and responsiveness of the service.
- Apply different prompt engineering principles to get better outputs.

I hope you enjoyed it!

Visit my [GitHub profile](https://github.com/rafaself).